# Data Cleaning — SaaS Raw Export

**Source data:** `data/raw/` — exported from production PostgreSQL database  
**Output:** `data/clean/` — validated and cleaned, ready for SQL analysis  
**Notebook:** 02 of 04

## Problems identified in raw data

| # | Problem | Tables affected |
|---|---------|----------------|
| 1 | Null values — missing country, channel, age, timestamps, plan | users, events, subscriptions |
| 2 | Duplicate rows — ETL pipeline ran twice during migration | all three tables |
| 3 | Inconsistent formatting — ORGANIC vs Organic vs organic | users, subscriptions |
| 4 | Invalid values — future dates, negative MRR, legacy event names | all three tables |
| 5 | Wrong data types — MRR stored as string, dates not parsed | users, events, subscriptions |

Each problem is inspected, documented, and fixed below.

In [1]:
import pandas as pd
import numpy as np

pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:.2f}'.format)

## Step 1 — Load Raw Data and Inspect


In [2]:
# Load raw files exactly as received — no changes yet

users  = pd.read_csv('data/raw/users.csv')
events = pd.read_csv('data/raw/events.csv')
subs   = pd.read_csv('data/raw/subscriptions.csv')

print("RAW DATA LOADED")
print("=" * 45)
print(f"  users         : {len(users):>10,} rows  {users.shape[1]} columns")
print(f"  events        : {len(events):>10,} rows  {events.shape[1]} columns")
print(f"  subscriptions : {len(subs):>10,} rows  {subs.shape[1]} columns")
print()

print("COLUMN NAMES AND DATA TYPES")
print("-" * 45)
print("\nusers:")
print(users.dtypes.to_string())
print("\nevents:")
print(events.dtypes.to_string())
print("\nsubscriptions:")
print(subs.dtypes.to_string())

RAW DATA LOADED
  users         :    102,000 rows  7 columns
  events        :    296,547 rows  5 columns
  subscriptions :     14,798 rows  7 columns

COLUMN NAMES AND DATA TYPES
---------------------------------------------

users:
user_id        object
signup_date    object
channel        object
device         object
country        object
age_band       object
plan_type      object

events:
event_id      object
user_id       object
event_name    object
event_ts      object
page          object

subscriptions:
sub_id         object
user_id        object
plan           object
mrr            object
start_date     object
end_date      float64
status         object


In [3]:
users.head()

,user_id,signup_date,channel,device,country,age_band,plan_type
0,u_0055972,2023-10-01,paid_search,desktop,US,45-54,free
1,u_0057675,2023-10-01,paid_search,mobile,GB,25-34,free
2,u_0071651,2023-10-01,organic,desktop,AU,45-54,free
3,u_0012343,2023-10-01,paid_social,desktop,AU,35-44,free
4,u_0070884,2023-10-01,referral,mobile,NaN,35-44,free


In [4]:
print("=" * 55)
print("  FULL INSPECTION REPORT — RAW DATA")
print("=" * 55)

#  USERS Table 
print("\n -------------- USERS --------------")

print(f"\n Shape : {users.shape}")

print(f"\n  Missing values:")
nulls = users.isnull().sum()
nulls = nulls[nulls > 0]
for col, n in nulls.items():
    print(f"    {col:<15}: {n:,}  ({n/len(users)*100:.1f}%)")

print(f"\n  Duplicate rows : {users.duplicated().sum():,}")

print(f"\n  signup_date dtype : {users['signup_date'].dtype}")

users_temp = pd.to_datetime(users['signup_date'], errors='coerce')
future     = (users_temp > pd.Timestamp('2024-10-31')).sum()
print(f"  Future dates : {future:,}")

print(f"\n  channel values :")
print(users['channel'].value_counts().head(10).to_string())

print(f"\n  device values :")
print(users['device'].value_counts().to_string())

  FULL INSPECTION REPORT — RAW DATA

 -------------- USERS --------------

 Shape : (102000, 7)

  Missing values:
    channel        : 3,061  (3.0%)
    device         : 63  (0.1%)
    country        : 5,089  (5.0%)
    age_band       : 2,042  (2.0%)

  Duplicate rows : 1,618

  signup_date dtype : object
  Future dates : 300

  channel values :
channel
organic          27865
paid_search      23333
paid_social      18631
referral         13815
email             9351
organic            603
Organic            599
ORGANIC            557
paid_search        510
PAID_SEARCH        494

  device values :
device
desktop      48825
mobile       37065
tablet       11794
Desktop        704
desktop        684
DESKTOP        657
mobile         535
Mobile         514
MOBILE         503
TABLET         165
tablet         162
Tablet         142
unknown         70
-               63
bot             54


In [5]:
events.head()

,event_id,user_id,event_name,event_ts,page
0,e_000000000,u_0055972,signup,2023-10-01 00:00:00,/signup
1,e_000000001,u_0057675,signup,2023-10-01 00:00:00,/signup
2,e_000000002,u_0071651,signup,2023-10-01 00:00:00,/signup
3,e_000000003,u_0071651,activation,2023-10-01 05:05:25,/onboarding/complete
4,e_000000004,u_0071651,feature_used,2023-10-01 19:53:16,/dashboard/feature


In [6]:
# EVENTS Table
print("\n-------------- EVENTS --------------")

print(f"\n  Shape              : {events.shape}")

print(f"\n  Missing values:")
nulls = events.isnull().sum()
nulls = nulls[nulls > 0]
for col, n in nulls.items():
    print(f"    {col:<15}: {n:,}  ({n/len(events)*100:.1f}%)")

print(f"\n  Duplicate rows : {events.duplicated().sum():,}")

valid_events = [
    'signup','activation','feature_used',
    'upgrade_initiated','paid_converted'
]
invalid_events = events[~events['event_name'].isin(valid_events)]
print(f"\n  Invalid event names: {len(invalid_events):,}")
print(invalid_events['event_name'].value_counts().to_string())

events_temp = pd.to_datetime(events['event_ts'], errors='coerce')
pre_launch  = (events_temp < pd.Timestamp('2023-10-01')).sum()
print(f"\n  Pre-launch events  : {pre_launch:,}")


-------------- EVENTS --------------

  Shape              : (296547, 5)

  Missing values:
    event_ts       : 2,956  (1.0%)

  Duplicate rows : 7,945

  Invalid event names: 800
event_name
legacy_event    220
page_view       206
btn_click       197
__test__        177

  Pre-launch events  : 400


In [7]:
subs.head()

,sub_id,user_id,plan,mrr,start_date,end_date,status
0,sub_0000000,u_0034091,pro,79,2023-10-04,NaN,active
1,sub_0000001,u_0011701,starter,27,2023-10-10,NaN,active
2,sub_0000002,u_0017829,starter,29,2023-10-09,NaN,active
3,sub_0000003,u_0025168,starter,30,2023-10-02,NaN,active
4,sub_0000004,u_0068979,pro,81,2023-10-02,NaN,active


In [8]:
#  SUBSCRIPTIONS 
print("\n-------------- SUBSCRIPTIONS --------------")

print(f"\n  Shape              : {subs.shape}")

print(f"\n  Missing values:")
nulls = subs.isnull().sum()
nulls = nulls[nulls > 0]
for col, n in nulls.items():
    print(f"    {col:<15}: {n:,}  ({n/len(subs)*100:.1f}%)")

print(f"\n  Duplicate rows     : {subs.duplicated().sum():,}")

print(f"\n  mrr dtype          : {subs['mrr'].dtype}")
print(f"  Sample mrr values  : {subs['mrr'].sample(6, random_state=1).tolist()}")

mrr_numeric  = pd.to_numeric(subs['mrr'], errors='coerce')
neg_zero_mrr = (mrr_numeric <= 0).sum()
print(f"\n  Negative/zero MRR  : {neg_zero_mrr:,}")

print(f"\n  plan values :")
print(subs['plan'].value_counts().to_string())

print("\n" + "=" * 55)
print("  END OF INSPECTION — begin cleaning below")
print("=" * 55)


-------------- SUBSCRIPTIONS --------------

  Shape              : (14798, 7)

  Missing values:
    plan           : 301  (2.0%)
    end_date       : 14,798  (100.0%)

  Duplicate rows     : 450

  mrr dtype          : object
  Sample mrr values  : ['28', '30', '27', '31', '79', '80']

  Negative/zero MRR  : 230

  plan values :
plan
starter       7792
pro           4893
business      1383
Starter         87
STARTER         81
starter         76
pro             53
Pro             49
PRO             46
BUSINESS        14
Business        13
business        10

  END OF INSPECTION — begin cleaning below


## Step 2 - Clean Users Table

Five cleaning operations applied in order:
1. Remove duplicate rows
2. Parse signup_date to datetime - fix wrong data type
3. Remove future dates - invalid values
4. Standardise channel, device, country - fix inconsistent formatting
5. Fill remaining null values

In [9]:
print("Cleaning users table...")
print(f"  Starting shape : {users.shape}")
users_df = users.copy()

# --------- 1. Remove duplicates ---------
before = len(users_df)
users_df = users_df.drop_duplicates()
print(f"\n  1. Duplicates removed     : {before - len(users_df):,}")

# --------- 2. Parse signup_date ---------
users_df['signup_date'] = pd.to_datetime(users_df['signup_date'], errors='coerce')
print(f"  2. signup_date dtype   : {users_df['signup_date'].dtype}")

# --------- 3. Remove future dates ---------
before = len(users_df)
users_df = users_df[users_df['signup_date'] <= pd.Timestamp('2024-10-31')]
print(f"  3. Future dates removed   : {before - len(users_df):,}")

# --------- 4. Standardise text columns ---------
users_df['channel']  = users_df['channel'].str.strip().str.lower()
users_df['device']   = users_df['device'].str.strip().str.lower()
users_df['country']  = users_df['country'].str.strip().str.upper()
users_df['age_band'] = users_df['age_band'].str.strip()
users_df['plan_type']= users_df['plan_type'].str.strip().str.lower()

# Remove rows where device is not a known valid value
valid_devices  = ['desktop', 'mobile', 'tablet']
valid_channels = ['organic','paid_search','paid_social','referral','email']

before = len(users_df)
users_df = users_df[users_df['device'].isin(valid_devices)]
print(f"  4a. Invalid device removed : {before - len(users_df):,}")

before = len(users_df)
users_df = users_df[users_df['channel'].isin(valid_channels) | users_df['channel'].isna()]
print(f"  4b. Invalid channel removed: {before - len(users_df):,}")

# --------- 5. Fill remaining nulls ---------
null_before = users_df.isnull().sum().sum()
users_df['country']  = users_df['country'].fillna('UNKNOWN')
users_df['channel']  = users_df['channel'].fillna('unknown')
users_df['age_band'] = users_df['age_band'].fillna('unknown')
null_after = users_df.isnull().sum().sum()
print(f"  5. Null values filled      : {null_before - null_after:,}")

print(f"\n  Final shape    : {users_df.shape}")
print(f"  Rows removed   : {len(users) - len(users_df):,}")
print(f"\n  channel values : {sorted(users_df['channel'].unique())}")
print(f"  device values  : {sorted(users_df['device'].unique())}")
print(f"\n  Date range     : {users_df['signup_date'].min().date()}"
      f" → {users_df['signup_date'].max().date()}")

Cleaning users table...
  Starting shape : (102000, 7)

  1. Duplicates removed     : 1,618
  2. signup_date dtype   : datetime64[ns]
  3. Future dates removed   : 300
  4a. Invalid device removed : 250
  4b. Invalid channel removed: 0
  5. Null values filled      : 9,981

  Final shape    : (99832, 7)
  Rows removed   : 2,168

  channel values : ['email', 'organic', 'paid_search', 'paid_social', 'referral', 'unknown']
  device values  : ['desktop', 'mobile', 'tablet']

  Date range     : 2023-10-01 → 2024-09-30


## Step 3 - Clean Events Table

Five cleaning operations applied in order:
1. Remove duplicate rows
2. Parse event_ts to datetime — fix wrong data type
3. Drop rows with missing timestamps — cannot place in funnel
4. Remove pre-launch events — impossible given product history
5. Remove legacy and unknown event names — not part of current funnel

In [11]:
print("Cleaning events table...")
print(f"  Starting shape : {events.shape}")
events_df = events.copy()

# --------- 1. Remove duplicates ---------
before = len(events_df)
events_df = events_df.drop_duplicates()
print(f"\n  1. Duplicates removed      : {before - len(events_df):,}")

# --------- 2. Parse event_ts ---------
events_df['event_ts'] = pd.to_datetime(events_df['event_ts'], errors='coerce')
print(f"  2. event_ts dtype           : {events_df['event_ts'].dtype}")

# --------- 3. Drop missing timestamps ---------
before = len(events_df)
events_df = events_df.dropna(subset=['event_ts'])
print(f"  3. Missing timestamps dropped : {before - len(events_df):,}")

# --------- 4. Remove pre-launch events ---------
before = len(events_df)
events_df = events_df[events_df['event_ts'] >= pd.Timestamp('2023-10-01')]
print(f"  4. Pre-launch events removed  : {before - len(events_df):,}")

# --------- 5. Keep only valid event names ---------
valid_events = [
    'signup','activation','feature_used',
    'upgrade_initiated','paid_converted'
]
before = len(events_df)
events_df = events_df[events_df['event_name'].isin(valid_events)]
print(f"  5. Invalid event names removed: {before - len(events_df):,}")

print(f"\n  Final shape    : {events_df.shape}")
print(f"  Rows removed   : {len(events) - len(events_df):,}")
print(f"\n  Event counts (clean):")
print(events_df['event_name'].value_counts().to_string())

Cleaning events table...
  Starting shape : (296547, 5)

  1. Duplicates removed      : 7,945
  2. event_ts dtype           : datetime64[ns]
  3. Missing timestamps dropped : 2,880
  4. Pre-launch events removed  : 400
  5. Invalid event names removed: 794

  Final shape    : (284528, 5)
  Rows removed   : 12,019

  Event counts (clean):
event_name
signup               98635
activation           75578
feature_used         64106
upgrade_initiated    32113
paid_converted       14096


## Step 4 - Clean Subscriptions Table

Five cleaning operations applied in order:
1. Remove duplicate rows
2. Fix MRR data type — strip $ symbol and convert to numeric
3. Remove zero and negative MRR — invalid revenue records
4. Standardise plan column — fix inconsistent formatting
5. Fill missing plan values — flag as unknown

In [12]:
print("Cleaning subscriptions table...")
print(f"  Starting shape : {subs.shape}")
subs_df = subs.copy()

# --------- 1. Remove duplicates ---------
before = len(subs_df)
subs_df = subs_df.drop_duplicates()
print(f"\n  1. Duplicates removed      : {before - len(subs_df):,}")

# --------- 2. Fix MRR data type ---------
subs_df['mrr'] = (
    subs_df['mrr']
    .astype(str)
    .str.replace('$', '', regex=False)
    .str.strip()
)
subs_df['mrr'] = pd.to_numeric(subs_df['mrr'], errors='coerce')
print(f"  2. mrr dtype now           : {subs_df['mrr'].dtype}")

# --------- 3. Remove invalid MRR ---------
before = len(subs_df)
subs_df = subs_df[subs_df['mrr'] > 0]
print(f"  3. Zero/negative MRR removed: {before - len(subs_df):,}")

# --------- 4. Standardise plan column ---------
subs_df['plan']   = subs_df['plan'].astype(str).str.strip().str.lower()
subs_df['status'] = subs_df['status'].astype(str).str.strip().str.lower()

valid_plans = ['starter', 'pro', 'business', 'nan', 'unknown']
print(f"  4. Plan values standardised")

# --------- 5. Fill missing plan ---------
subs_df['plan'] = subs_df['plan'].replace('nan', 'unknown')
null_plan  = (subs_df['plan'] == 'unknown').sum()
print(f"  5. Missing plan flagged    : {null_plan:,} rows → 'unknown'")

# Parse start_date
subs_df['start_date'] = pd.to_datetime(subs_df['start_date'], errors='coerce')

print(f"\n  Final shape    : {subs_df.shape}")
print(f"  Rows removed   : {len(subs) - len(subs_df):,}")

print(f"\n  MRR stats:")
print(f"    Min  : ${subs_df['mrr'].min():.0f}")
print(f"    Max  : ${subs_df['mrr'].max():.0f}")
print(f"    Mean : ${subs_df['mrr'].mean():.2f}")
print(f"    Total: ${subs_df['mrr'].sum():,.0f}")

print(f"\nPlan distribution:")
print(subs_df['plan'].value_counts().to_string())

Cleaning subscriptions table...
  Starting shape : (14798, 7)

  1. Duplicates removed      : 450
  2. mrr dtype now           : int64
  3. Zero/negative MRR removed: 230
  4. Plan values standardised
  5. Missing plan flagged    : 281 rows → 'unknown'

  Final shape    : (14118, 7)
  Rows removed   : 680

  MRR stats:
    Min  : $27
    Max  : $201
    Mean : $62.92
    Total: $888,305

Plan distribution:
plan
starter     7682
pro         4802
business    1353
unknown      281


## Step 5 - Final Validation 

Cross-table checks before saving:
- Every user_id in subscriptions exists in users
- Every user_id in events exists in users
- No nulls remain in critical columns
- Row counts are reasonable

In [13]:
print("=" * 55)
print("  FINAL VALIDATION")
print("=" * 55)

# --------- Cross-table integrity checks ---------
valid_user_ids = set(users_df['user_id'].unique())

orphan_events = (~events_df['user_id'].isin(valid_user_ids)).sum()
orphan_subs   = (~subs_df['user_id'].isin(valid_user_ids)).sum()

print(f"\n  Events with no matching user  : {orphan_events:,}")
print(f"  Subs with no matching user    : {orphan_subs:,}")

# Remove orphans if any exist
if orphan_events > 0:
    events_df = events_df[events_df['user_id'].isin(valid_user_ids)]
    print(f"  → Orphan events removed")

if orphan_subs > 0:
    subs_df = subs_df[subs_df['user_id'].isin(valid_user_ids)]
    print(f"  → Orphan subs removed")

# --------- Null check on critical columns ---------
print(f"\n  Remaining nulls in critical columns:")
critical = {
    'users'         : (users_df, ['user_id','signup_date','channel','device']),
    'events'        : (events_df, ['event_id','user_id','event_name','event_ts']),
    'subscriptions' : (subs_df, ['sub_id','user_id','mrr','plan']),
}
all_clean = True
for table_name, (df, cols) in critical.items():
    for col in cols:
        n = df[col].isna().sum()
        if n > 0:
            print(f"    {table_name}.{col} : {n:,} nulls remaining")
            all_clean = False
if all_clean:
    print(f"    None — all critical columns are complete")

  FINAL VALIDATION

  Events with no matching user  : 1,458
  Subs with no matching user    : 73
  → Orphan events removed
  → Orphan subs removed

  Remaining nulls in critical columns:
    None — all critical columns are complete


In [14]:
events_df[events_df['event_name'] == 'signup']['user_id'].count()

98118

In [15]:
events_df['user_id'].nunique()

99136

## Step 6 — Enforce Funnel Integrity

A valid funnel requires every user in the events table
to have a confirmed signup event.

Users without a signup event cannot be placed at the
top of the funnel — their conversion rates cannot be
calculated accurately.

Action: Remove all events belonging to users who have
no signup event row. This affects downstream data only —
the users table is not changed.

In [16]:
print("Enforcing funnel integrity...")
print(f"  Events before : {len(events_df):,}")
print(f"  Users before  : {events_df['user_id'].nunique():,}")

# Find users who have a confirmed signup event
users_with_signup = set(
    events_df[events_df['event_name'] == 'signup']['user_id'].unique()
)

print(f"\n  Users WITH signup event    : {len(users_with_signup):,}")
print(f"  Users WITHOUT signup event : "
      f"{events_df['user_id'].nunique() - len(users_with_signup):,}")

# Remove ALL events for users who have no signup event
# This keeps the funnel clean — every user starts at signup
events_df = events_df[events_df['user_id'].isin(users_with_signup)]

print(f"\n  Events after  : {len(events_df):,}")
print(f"  Users after   : {events_df['user_id'].nunique():,}")
print(f"  Rows removed  : {len(events) - len(events_df):,}")

# Verify — should now be zero
remaining = events_df['user_id'].nunique() - \
            events_df[events_df['event_name'] == 'signup']['user_id'].nunique()
print(f"\n  Users without signup: {remaining}")

Enforcing funnel integrity...
  Events before : 283,070
  Users before  : 99,136

  Users WITH signup event    : 98,118
  Users WITHOUT signup event : 1,018

  Events after  : 280,594
  Users after   : 98,118
  Rows removed  : 15,953

  Users without signup: 0


## Final summary

In [17]:
# --------- Final summary ---------
print("=" * 55)
print(" FINAL CLEANING SUMMARY ")
print("=" * 55)
print(f"  {'TABLE':<16} {'RAW':>10} {'CLEAN':>10} {'REMOVED':>10}")
print(f"  {'-'*48}")
print(f"  {'users':<16} {len(users):>10,} {len(users_df):>10,} "
      f"{len(users)-len(users_df):>10,}")
print(f"  {'events':<16} {len(events):>10,} {len(events_df):>10,} "
      f"{len(events)-len(events_df):>10,}")
print(f"  {'subscriptions':<16} {len(subs):>10,} {len(subs_df):>10,} "
      f"{len(subs)-len(subs_df):>10,}")
print("=" * 55)


# --------- Save clean files ---------
users_df.to_csv('data/clean/users_clean.csv', index=False)
events_df.to_csv('data/clean/events_clean.csv', index=False)
subs_df.to_csv('data/clean/subscriptions_clean.csv', index=False)

print(f"\n  Clean files saved to data/clean/")
print("=" * 55)

 FINAL CLEANING SUMMARY 
  TABLE                   RAW      CLEAN    REMOVED
  ------------------------------------------------
  users               102,000     99,832      2,168
  events              296,547    280,594     15,953
  subscriptions        14,798     14,045        753



  Clean files saved to data/clean/
